In [22]:
import os
import sys
import json
import h5py

import numpy as np
import pandas as pd
sys.path.insert(0, '../data_processing/palette/')
sys.path.insert(0, '../data_processing/util/')

from palette import Palette
from util import array_to_schematic


In [23]:
data_path = '../data/'

old_data_path = os.path.join(data_path, 'processed_builds/')
new_data_path = os.path.join(data_path, 'new_data/java/')
embeddings_path = os.path.join(data_path, 'embeddings_newline.json')
block_dist_path = os.path.join(data_path, 'block_dist_dataframe.csv')


old_data_filenames = [f for f in os.listdir(old_data_path) if f.endswith('.h5')]
new_data_filenames = os.listdir(new_data_path)

old_data_token2block_path = os.path.join(data_path, 'tok2block.json')
new_data_block2token_path = os.path.join(data_path, 'java_palette.json')

with open(old_data_token2block_path) as f:
    old_data_token2block = json.load(f)
old_data_block2token = {v:int(k) for k,v in old_data_token2block.items()}
del old_data_block2token["UNKNOWN_BLOCK"]

with open(new_data_block2token_path) as f:
    new_data_block2token = json.load(f)

with open(embeddings_path) as f:
    embeddings = json.load(f)
embedded_blocks = list(embeddings.keys())
embedded_blocks.remove("UNKNOWN_BLOCK")


block_dist_df = pd.read_csv(block_dist_path)

dataset_path = os.path.join(data_path, 'MinecraftVAEDataset/')
dataset_samples_path = os.path.join(dataset_path, 'samples/')
dataset_schems_path = os.path.join(dataset_path, 'schems/')

In [24]:
old_data_palette = Palette(old_data_block2token)
new_data_palette = Palette(new_data_block2token)
embeddings_palette = Palette(embedded_blocks)

emb_lookup = np.zeros((len(embedded_blocks), 32), dtype=np.float64)
for block in embedded_blocks:
    emb_lookup[embeddings_palette.block2token[block]] = embeddings[block]

Added 87 blocks for potential tranformations
Added 556 blocks for potential tranformations
Added 87 blocks for potential tranformations


#### Palette Simplifiers

In [25]:
keep_blockstates = ["axis", "facing", "shape"]

block_ids_strip = [
    "minecraft:bed",
    "minecraft:black_shulker_box",
    "minecraft:chest",
    "minecraft:deepslate",
    "minecraft:quartz_pillar",
    "minecraft:deepslate",
    "minecraft:repeater",
    "minecraft:wall_torch",
    "minecraft:rail",
    "minecraft:acacia_button",
    "minecraft:acacia_fence_gate",
    "minecraft:bamboo_button",
    "minecraft:bamboo_fence_gate",
    "minecraft:birch_button",
    "minecraft:birch_fence_gate",
    "minecraft:cherry_button",
    "minecraft:cherry_fence_gate",
    "minecraft:chipped_anvil",
    "minecraft:crimson_button",
    "minecraft:crimson_fence_gate",
    "minecraft:damaged_anvil",
    "minecraft:dark_oak_button",
    "minecraft:dark_oak_fence_gate",
    "minecraft:grindstone",
    "minecraft:jungle_button",
    "minecraft:jungle_fence_gate",
    "minecraft:ladder",
    "minecraft:mangrove_button",
    "minecraft:mangrove_fence_gate",
    "minecraft:oak_button",
    "minecraft:oak_fence_gate",
    "minecraft:polished_blackstone_button",
    "minecraft:spruce_button",
    "minecraft:spruce_fence_gate",
    "minecraft:stone_button",
    "minecraft:stonecutter",
    "minecraft:tripwire_hook",
    "minecraft:warped_button",
    "minecraft:warped_fence_gate"
]

#### Reduce the Blockstates

In [26]:
# Reduce the blockstates of new dataset
reduced_new_palette, new_src2tgt_1, new_data_tgt2src_2 = new_data_palette.reduce_blockstates(keep_blockstates)
reduced_old_palette, old_src2tgt_1, old_data_tgt2src_2 = old_data_palette.reduce_blockstates(keep_blockstates)
reduced_emb_palette, emb_src2tgt_1, emb_data_tgt2src_2 = embeddings_palette.reduce_blockstates(keep_blockstates)

# Reduce specific blocks' blockstates in new dataset
final_new_palette, new_src2tgt_2, new_data_tgt2src_1 = reduced_new_palette.reduce_blockstates(keep_blockstates=[], block_ids=block_ids_strip)
final_old_palette, old_src2tgt_2, old_data_tgt2src_1 = reduced_old_palette.reduce_blockstates(keep_blockstates=[], block_ids=block_ids_strip)
final_emb_palette, emb_src2tgt_2, emb_data_tgt2src_1 = reduced_emb_palette.reduce_blockstates(keep_blockstates=[], block_ids=block_ids_strip)

emb_palette_no_blockstate, emb2blockids, blockids2emb = final_emb_palette.reduce_blockstates(keep_blockstates=[])

Added 0 blocks for potential tranformations
Added 0 blocks for potential tranformations
Added 0 blocks for potential tranformations
Added 0 blocks for potential tranformations
Added 0 blocks for potential tranformations
Added 0 blocks for potential tranformations
Added 0 blocks for potential tranformations


#### Exclude low-occuring blocks from new dataset palette to closer match the embeddings palette

In [15]:
threshold = block_dist_df["total_occurences"].quantile(0.75)
mask_blocks = block_dist_df[block_dist_df["total_occurences"] < threshold]
mask_list = mask_blocks["Block"].to_list()

# Mask low-occuring blocks in new dataset
new_data_mask = final_new_palette.get_mask_lookup(mask_list)
new_data_unmasked_blocks = list(set([final_new_palette.token2block[i] for i, token in enumerate(new_data_mask) if i == token]))
# new_data_palette = Palette(new_data_unmasked_blocks)

#### Map the old and new dataset palettes to the embeddings

In [16]:
old_to_emb_lookup = np.ones(len(final_old_palette), dtype=np.int16) * -1
new_to_emb_lookup = np.ones(len(final_new_palette), dtype=np.int16) * -1

for block in final_old_palette.block_strings:
    old_to_emb_lookup[final_old_palette.block2token[block]] = final_emb_palette.block2token[block]

for block in new_data_unmasked_blocks:
    new_to_emb_lookup[final_new_palette.block2token[block]] = final_emb_palette.block2token[block]

In [ ]:
emb_to_game_lookup = np.arange(len(final_emb_palette))

#### Process the old and new dataset to match the embeddings

In [ ]:
embedded_block_dist_df = pd.DataFrame(columns=["Block", "total_occurences", "sample_occurences"])

block_counts_per_sample = np.zeros((len(old_data_filenames) + len(new_data_filenames), len(emb_palette_no_blockstate)), dtype=np.int32)

block_counts = np.zeros(len(emb_palette_no_blockstate), dtype=np.int16)

sample_dims_df = pd.DataFrame(columns=["Sample", "x", "y", "z", "volume"])

for i, file in enumerate(old_data_filenames):
    path = os.path.join(old_data_path, file)
    with h5py.File(path, 'r') as f:
        arr = np.array(f[file][:], dtype=np.int16)
        arr = old_src2tgt_1[arr]
        arr = old_src2tgt_2[arr]
        arr = old_to_emb_lookup[arr]
        
        arr_to_schem = emb_data_tgt2src_1[arr]
        arr_to_schem = emb_data_tgt2src_2[arr_to_schem]
        
        blocks, counts = np.unique(emb2blockids[arr], return_counts=True)
        
        #-- Add counts to block_occurences
        block_counts[blocks] += counts
        block_counts_this_sample = np.zeros(len(emb_palette_no_blockstate), dtype=np.int64)
        block_counts_this_sample[blocks] += counts
        block_counts_per_sample[i] = block_counts_this_sample
        
        # Save outputs as npy and schem
        sample_name = file.removesuffix('.h5')
        np.save(os.path.join(dataset_samples_path, sample_name), arr)
        
        array_to_schematic(arr_to_schem, embeddings_palette.token2block, dataset_schems_path, sample_name)
        
        row = {
            "Sample": sample_name,
            "x": arr.shape[0],
            "y": arr.shape[1],
            "z": arr.shape[2],
            "volume": arr.shape[0] * arr.shape[1] * arr.shape[2]
        }
        
        sample_dims_df.loc[len(sample_dims_df)] = row
        
for k, file in enumerate(new_data_filenames):
    path = os.path.join(new_data_path, file)
    arr = np.load(path)
    arr = new_src2tgt_1[arr]
    arr = new_src2tgt_2[arr]
    arr = new_data_mask[arr]
    arr = new_to_emb_lookup[arr]
    
    arr_to_schem = emb_data_tgt2src_1[arr]
    arr_to_schem = emb_data_tgt2src_2[arr_to_schem]
    
    blocks, counts = np.unique(emb2blockids[arr], return_counts=True)
    
    #-- Add counts to block_occurences
    block_counts[blocks] += counts
    block_counts_this_sample = np.zeros(len(emb_palette_no_blockstate), dtype=np.int64)
    block_counts_this_sample[blocks] += counts
    block_counts_per_sample[len(old_data_filenames)+k] = block_counts_this_sample
    
    # Save outputs as npy and schem
    sample_name = file.removesuffix('_java.npy')
    np.save(os.path.join(dataset_samples_path, sample_name), arr)
    
    array_to_schematic(arr_to_schem, embeddings_palette.token2block, dataset_schems_path, sample_name)
    
    row = {
        "Sample": sample_name,
        "x": arr.shape[0],
        "y": arr.shape[1],
        "z": arr.shape[2],
        "volume": arr.shape[0] * arr.shape[1] * arr.shape[2]
    }
    
    sample_dims_df.loc[len(sample_dims_df)] = row


block_counts_per_sample_df = pd.DataFrame(block_counts_per_sample, index=old_data_filenames+new_data_filenames, columns=emb_palette_no_blockstate.block_strings)
block_counts_per_sample_df = block_counts_per_sample_df.reset_index(names="Sample")


Successfully saved schematic to build_batch_100_2574_1.schem
Successfully saved schematic to build_batch_100_2578_1.schem
Successfully saved schematic to build_batch_100_2592_1.schem
Successfully saved schematic to build_batch_100_2598_1.schem
Successfully saved schematic to build_batch_101_2600_1.schem
Successfully saved schematic to build_batch_101_2602_1.schem
Successfully saved schematic to build_batch_101_2603_1.schem
Successfully saved schematic to build_batch_101_2605_1.schem
Successfully saved schematic to build_batch_101_2606_1.schem
Successfully saved schematic to build_batch_101_2610_1.schem
Successfully saved schematic to build_batch_101_2611_1.schem
Successfully saved schematic to build_batch_101_2616_1.schem
Successfully saved schematic to build_batch_101_2618_1.schem
Successfully saved schematic to build_batch_101_2619_1.schem
Successfully saved schematic to build_batch_102_2628_1.schem
Successfully saved schematic to build_batch_102_2629_1.schem
Successfully saved schem

In [17]:
final_emb_to_vec = np.arange(len(final_emb_palette), dtype=np.int64)

In [18]:
final_emb_to_vec = emb_data_tgt2src_1[final_emb_to_vec]
final_emb_to_vec = emb_data_tgt2src_2[final_emb_to_vec]
emb2game_lookup = final_emb_to_vec
final_emb_to_vec = emb_lookup[final_emb_to_vec]

In [19]:
final_emb_to_vec.shape

(2352, 32)

In [20]:
np.save(os.path.join(dataset_path, "token2vector.npy"), final_emb_to_vec)
np.save(os.path.join(dataset_path, "emb2game_lookup.npy"), emb2game_lookup)

In [21]:
with open(os.path.join(dataset_path, 'block2token.json'), "w") as f:
    json.dump(final_emb_palette.block2token, fp=f, indent=4)
with open(os.path.join(dataset_path, 'game_block2token.json'), "w") as f:
    json.dump(embeddings_palette.block2token, fp=f, indent=4)

In [41]:
vectors, counts = np.unique(final_emb_to_vec, return_counts=True, axis=0)


In [42]:
np.unique(counts)

array([1, 2], dtype=int64)

In [26]:
# save to file
# block_counts_data = {emb_palette_no_blockstate.token2block[i]: [count] for i, count in enumerate(block_counts)}
# block_counts_df = pd.DataFrame.from_dict(block_counts_data, orient='index', columns=["count"])
# block_counts_df.index.name = "block"

block_counts_per_sample_df.to_csv(os.path.join(dataset_path, "block_counts_per_sample_df.csv"))
sample_dims_df.to_csv(os.path.join(dataset_path, "sample_dims_df.csv"))


In [ ]:
test = np.array([2,4,4,2,3,5,25,2,2,33,3,3,3,2])
np.unique(test, return_counts=True)

(array([ 2,  3,  4,  5, 25, 33]), array([5, 4, 2, 1, 1, 1], dtype=int64))